In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image.

import os
import gc
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
!pip install -q dagshub mlflow

import dagshub
import mlflow
import mlflow.sklearn

dagshub.init(repo_owner="ChorniBero15", repo_name="ML2", mlflow=True)

mlflow.set_experiment("XGBoost_Training")

REGISTERED_MODEL_NAME = "model_ieee_cis_xgboost"


# 1. Cleaning


In [ ]:
def reduce_mem_usage(df, use_category=False):
    start_mem = df.memory_usage(deep=True).sum() / 1024 ** 2
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if pd.api.types.is_integer_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="integer")
        
        elif pd.api.types.is_float_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="float")
        
        elif use_category and col_type == "object":
            unique_ratio = df[col].nunique(dropna=False) / len(df)
            
            if unique_ratio < 0.5:
                df[col] = df[col].astype("category")
    
    end_mem = df.memory_usage(deep=True).sum() / 1024 ** 2
    reduction = 100 * (start_mem - end_mem) / start_mem
    
    print(f"Memory usage: {start_mem:.2f} MB -> {end_mem:.2f} MB")
    print(f"Reduced by {reduction:.2f}%")
    
    return df

In [ ]:
train_transaction_path = "/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv"
train_identity_path = "/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv"

test_identity_path = "/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv"
test_transaction_path = "/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv"

train_transaction = reduce_mem_usage(pd.read_csv(train_transaction_path))
train_identity = reduce_mem_usage(pd.read_csv(train_identity_path))

test_transaction = reduce_mem_usage(pd.read_csv(test_transaction_path))
test_identity = reduce_mem_usage(pd.read_csv(test_identity_path))

test_identity.columns = test_identity.columns.str.replace("-", "_", regex=False)
test_transaction.columns = test_transaction.columns.str.replace("-", "_", regex=False)

train = train_transaction.merge(train_identity, on="TransactionID", how="left")
test = test_transaction.merge(test_identity, on="TransactionID", how="left")

print("Merged train:", train.shape)
print("Merged test:", test.shape)

import gc

del train_transaction, test_transaction, train_identity, test_identity
gc.collect()

train = reduce_mem_usage(train)
test = reduce_mem_usage(test)

In [ ]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    mlflow.log_param("cleaning_method", "memory_downcasting_and_left_merge")
    mlflow.log_param("merge_key", "TransactionID")
    mlflow.log_param("merge_type", "left")
    mlflow.log_metric("train_rows", train.shape[0])
    mlflow.log_metric("train_columns", train.shape[1])
    mlflow.log_metric("test_rows", test.shape[0])
    mlflow.log_metric("test_columns", test.shape[1])
    mlflow.log_metric("fraud_rate", train["isFraud"].mean())
    mlflow.log_metric("train_missing_percent", train.isnull().mean().mean() * 100)
    mlflow.log_metric("test_missing_percent", test.isnull().mean().mean() * 100)
    mlflow.log_metric("train_duplicate_transaction_ids", train["TransactionID"].duplicated().sum())
    mlflow.log_metric("test_duplicate_transaction_ids", test["TransactionID"].duplicated().sum())

print("Cleaning metrics logged to MLflow.")


# 2. Feature Engineering


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from xgboost import XGBClassifier


In [ ]:
class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None):
        self.cols = cols if cols is not None else []
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        existing_cols = [col for col in self.cols if col in X.columns]
        return X.drop(columns=existing_cols)

In [ ]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.created_cols = [
            "TransactionAmt_log",
            "TransactionAmt_decimal",
            "Transaction_day",
            "Transaction_hour",
            "Transaction_week",
            "missing_count",
            "has_identity",
            "email_domain_match",
            "card_missing_count"
        ]
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        X = X.drop(columns=[col for col in self.created_cols if col in X.columns], errors="ignore")
        
        features = pd.DataFrame(index=X.index)
        
        if "TransactionAmt" in X.columns:
            features["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
            features["TransactionAmt_decimal"] = ((X["TransactionAmt"] % 1) * 1000).round()
        
        if "TransactionDT" in X.columns:
            features["Transaction_day"] = X["TransactionDT"] // (24 * 60 * 60)
            features["Transaction_hour"] = (X["TransactionDT"] // (60 * 60)) % 24
            features["Transaction_week"] = features["Transaction_day"] // 7
        
        features["missing_count"] = X.isnull().sum(axis=1)
        
        if "id_01" in X.columns:
            features["has_identity"] = X["id_01"].notnull().astype("int8")
        else:
            features["has_identity"] = 0
        
        if "P_emaildomain" in X.columns and "R_emaildomain" in X.columns:
            features["email_domain_match"] = (X["P_emaildomain"] == X["R_emaildomain"]).astype("int8")
        else:
            features["email_domain_match"] = 0
        
        card_cols = [col for col in ["card1", "card2", "card3", "card4", "card5", "card6"] if col in X.columns]
        
        if len(card_cols) > 0:
            features["card_missing_count"] = X[card_cols].isnull().sum(axis=1)
        else:
            features["card_missing_count"] = 0
        
        return pd.concat([X, features], axis=1)

In [ ]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None):
        self.cols = cols
        self.freq_maps = {}
    
    def _normalize_series(self, s):
        return s.astype("object").where(s.notnull(), "__MISSING__")
    
    def fit(self, X, y=None):
        if self.cols is None:
            self.cols_ = X.select_dtypes(include=["object", "category"]).columns.tolist()
        else:
            self.cols_ = [col for col in self.cols if col in X.columns]
        
        self.freq_maps = {}
        
        for col in self.cols_:
            normalized = self._normalize_series(X[col])
            self.freq_maps[col] = normalized.value_counts(dropna=False).to_dict()
        
        return self
    
    def transform(self, X):
        X = X.copy()
        new_features = pd.DataFrame(index=X.index)
        
        for col in self.cols_:
            if col in X.columns:
                normalized = self._normalize_series(X[col])
                new_features[f"{col}_freq"] = normalized.map(self.freq_maps[col]).fillna(0).astype("float32")
        
        return pd.concat([X, new_features], axis=1)


In [ ]:
class CategoricalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None):
        self.cols = cols
        self.category_maps = {}
    
    def _normalize_series(self, s):
        return s.astype("object").where(s.notnull(), "__MISSING__")
    
    def fit(self, X, y=None):
        if self.cols is None:
            self.cols_ = X.select_dtypes(include=["object", "category"]).columns.tolist()
        else:
            self.cols_ = [col for col in self.cols if col in X.columns]
        
        self.category_maps = {}
        
        for col in self.cols_:
            normalized = self._normalize_series(X[col])
            unique_values = pd.Series(normalized.unique())
            self.category_maps[col] = {value: idx for idx, value in enumerate(unique_values)}
        
        return self
    
    def transform(self, X):
        X = X.copy()
        
        for col in self.cols_:
            if col in X.columns:
                normalized = self._normalize_series(X[col])
                X[col] = normalized.map(self.category_maps[col]).fillna(-1).astype("int32")
        
        return X


In [ ]:
class ReplaceInfValues(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        X = X.replace([np.inf, -np.inf], np.nan)
        return X

In [ ]:
class SimpleFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, max_missing_ratio=0.95, min_unique_values=2):
        self.max_missing_ratio = max_missing_ratio
        self.min_unique_values = min_unique_values

    def fit(self, X, y=None):
        X = X.copy()
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        X_numeric = X[numeric_cols]

        missing_ratio = X_numeric.isnull().mean()
        unique_counts = X_numeric.nunique(dropna=False)

        self.selected_features_ = [
            col for col in numeric_cols
            if missing_ratio[col] <= self.max_missing_ratio
            and unique_counts[col] >= self.min_unique_values
        ]
        self.dropped_features_ = [col for col in numeric_cols if col not in self.selected_features_]

        return self

    def transform(self, X):
        X = X.copy()

        for col in self.selected_features_:
            if col not in X.columns:
                X[col] = np.nan

        return X[self.selected_features_]


# 3. Validation Split


In [ ]:
target = "isFraud"

X = train.drop(columns=[target])
y = train[target].astype("int8")

X_test = test.copy()

print("X:", X.shape)
print("y:", y.shape)
print("X_test:", X_test.shape)

In [ ]:
split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index].copy()
y_train = y.iloc[:split_index].copy()

X_val = X.iloc[split_index:].copy()
y_val = y.iloc[split_index:].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

print("Train fraud rate:", y_train.mean())
print("Validation fraud rate:", y_val.mean())

In [ ]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Negative count:", negative_count)
print("Positive count:", positive_count)
print("scale_pos_weight:", scale_pos_weight)

In [ ]:
freq_encode_cols = [
    "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2",
    "P_emaildomain", "R_emaildomain",
    "DeviceType", "DeviceInfo",
    "ProductCD",
    "id_30", "id_31", "id_33"
]

xgb_params = {
    "n_estimators": 1000,
    "learning_rate": 0.03,
    "max_depth": 5,
    "min_child_weight": 10,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "gamma": 0.1,
    "reg_alpha": 0.1,
    "reg_lambda": 2.0,
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
    "scale_pos_weight": scale_pos_weight
}

xgb_pipeline = Pipeline([
    ("drop_columns", DropColumns(cols=["TransactionID"])),
    ("feature_engineering", FeatureEngineering()),
    ("frequency_encoding", FrequencyEncoder(cols=freq_encode_cols)),
    ("categorical_encoding", CategoricalEncoder()),
    ("replace_inf", ReplaceInfValues()),
    ("feature_selection", SimpleFeatureSelector(max_missing_ratio=0.95, min_unique_values=2)),
    ("model", XGBClassifier(**xgb_params))
])

In [ ]:
with mlflow.start_run(run_name="XGBoost_Feature_Engineering"):
    fe_pipeline = Pipeline([
        ("drop_columns", DropColumns(cols=["TransactionID"])),
        ("feature_engineering", FeatureEngineering()),
        ("frequency_encoding", FrequencyEncoder(cols=freq_encode_cols)),
        ("categorical_encoding", CategoricalEncoder()),
        ("replace_inf", ReplaceInfValues())
    ])

    fe_pipeline.fit(X_train, y_train)
    X_train_fe_sample = fe_pipeline.transform(X_train.iloc[:2000])

    mlflow.log_param("created_features", "TransactionAmt_log,TransactionAmt_decimal,Transaction_day,Transaction_hour,Transaction_week,missing_count,has_identity,email_domain_match,card_missing_count")
    mlflow.log_param("categorical_encoding", "ordinal_codes_fitted_on_training_split")
    mlflow.log_param("frequency_encoding_columns", ",".join(freq_encode_cols))
    mlflow.log_metric("features_after_engineering", X_train_fe_sample.shape[1])
    mlflow.log_metric("sample_rows_for_feature_engineering_check", X_train_fe_sample.shape[0])

print("Feature engineering metrics logged to MLflow.")


# 4. Training


# 4.1. Grid Search for XGBoost Hyperparameters


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.base import BaseEstimator, TransformerMixin

from xgboost import XGBClassifier

import json
import mlflow
import numpy as np
import pandas as pd

In [ ]:
xgb_param_grid = {
    "model__n_estimators": [600, 1000],
    "model__learning_rate": [0.03, 0.04],
    "model__max_depth": [5],
    "model__min_child_weight": [8, 10],
    "model__subsample": [0.85],
    "model__colsample_bytree": [0.85],
    "model__gamma": [0.1],
    "model__reg_alpha": [0.1],
    "model__reg_lambda": [2.0]
}

candidate_count = 1
for values in xgb_param_grid.values():
    candidate_count *= len(values)

print("Grid candidate count:", candidate_count)

grid_search_pipeline = Pipeline([
    ("drop_columns", DropColumns(cols=["TransactionID"])),
    ("feature_engineering", FeatureEngineering()),
    ("frequency_encoding", FrequencyEncoder(cols=freq_encode_cols)),
    ("categorical_encoding", CategoricalEncoder()),
    ("replace_inf", ReplaceInfValues()),
    ("feature_selection", SimpleFeatureSelector(max_missing_ratio=0.95, min_unique_values=2)),
    ("model", XGBClassifier(**xgb_params))
])

time_cv = TimeSeriesSplit(n_splits=2)

grid_search = GridSearchCV(
    estimator=grid_search_pipeline,
    param_grid=xgb_param_grid,
    scoring="roc_auc",
    cv=time_cv,
    n_jobs=1,
    verbose=2,
    return_train_score=True,
    refit=True
)

with mlflow.start_run(run_name="XGBoost_Grid_Search"):
    grid_search.fit(X_train, y_train)

    grid_results_df = pd.DataFrame(grid_search.cv_results_)
    grid_results_df.to_csv("xgboost_grid_search_results.csv", index=False)

    train_pred_proba = grid_search.best_estimator_.predict_proba(X_train)[:, 1]
    val_pred_proba = grid_search.best_estimator_.predict_proba(X_val)[:, 1]

    grid_train_roc_auc = roc_auc_score(y_train, train_pred_proba)
    grid_val_roc_auc = roc_auc_score(y_val, val_pred_proba)
    grid_train_pr_auc = average_precision_score(y_train, train_pred_proba)
    grid_val_pr_auc = average_precision_score(y_val, val_pred_proba)
    grid_overfit_gap = grid_train_roc_auc - grid_val_roc_auc

    with open("xgboost_param_grid.json", "w") as f:
        json.dump(xgb_param_grid, f, indent=2)

    mlflow.log_param("search_method", "GridSearchCV")
    mlflow.log_param("cv_strategy", "TimeSeriesSplit")
    mlflow.log_param("cv_splits", time_cv.n_splits)
    mlflow.log_param("scoring", "roc_auc")
    mlflow.log_param("candidate_count", candidate_count)

    for param_name, param_value in grid_search.best_params_.items():
        mlflow.log_param(f"best_{param_name}", param_value)

    mlflow.log_metric("best_cv_roc_auc", grid_search.best_score_)
    mlflow.log_metric("grid_train_roc_auc", grid_train_roc_auc)
    mlflow.log_metric("grid_validation_roc_auc", grid_val_roc_auc)
    mlflow.log_metric("grid_train_pr_auc", grid_train_pr_auc)
    mlflow.log_metric("grid_validation_pr_auc", grid_val_pr_auc)
    mlflow.log_metric("grid_overfit_gap", grid_overfit_gap)

    mlflow.log_artifact("xgboost_param_grid.json")
    mlflow.log_artifact("xgboost_grid_search_results.csv")

best_params_from_grid = xgb_params.copy()

for param_name, param_value in grid_search.best_params_.items():
    clean_name = param_name.replace("model__", "")
    best_params_from_grid[clean_name] = param_value

grid_pipeline = grid_search.best_estimator_
xgb_pipeline = grid_pipeline
best_params_for_final = best_params_from_grid.copy()
best_params_source = "XGBoost_Grid_Search"
best_experiment_name = "XGBoost_Grid_Search"

grid_metrics = {
    "run_name": "XGBoost_Grid_Search",
    "params_source": "grid_search_best_estimator",
    "best_cv_roc_auc": grid_search.best_score_,
    "train_roc_auc": grid_train_roc_auc,
    "validation_roc_auc": grid_val_roc_auc,
    "train_pr_auc": grid_train_pr_auc,
    "validation_pr_auc": grid_val_pr_auc,
    "overfit_gap": grid_overfit_gap
}

print("Best grid search params:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_)
print("Grid validation ROC-AUC:", grid_val_roc_auc)
print("Grid validation PR-AUC:", grid_val_pr_auc)
print("Grid overfit gap:", grid_overfit_gap)

In [ ]:
val_pred_proba = xgb_pipeline.predict_proba(X_val)[:, 1]
val_pred = (val_pred_proba >= 0.5).astype(int)

val_roc_auc = roc_auc_score(y_val, val_pred_proba)
val_pr_auc = average_precision_score(y_val, val_pred_proba)

print("Selected model:", best_experiment_name)
print("Selected validation ROC-AUC:", val_roc_auc)
print("Selected validation PR-AUC:", val_pr_auc)

print("\nClassification report:")
print(classification_report(y_val, val_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_val, val_pred))


# 5. Feature Selection


In [ ]:
preprocessor = xgb_pipeline[:-1]
model = xgb_pipeline.named_steps["model"]

sample_transformed = preprocessor.transform(X_train.iloc[:5])
feature_names = sample_transformed.columns.tolist()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
})

importance_df = importance_df.sort_values("importance", ascending=False)

importance_df.head(30)

In [ ]:
import matplotlib.pyplot as plt

top_importance = importance_df.head(30)

plt.figure(figsize=(10, 8))
plt.barh(top_importance["feature"][::-1], top_importance["importance"][::-1])
plt.title("Top 30 XGBoost Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

In [ ]:
selected_top_features = importance_df.head(100)["feature"].tolist()

importance_df.to_csv("xgboost_feature_importance.csv", index=False)
with open("xgboost_top_100_features.json", "w") as f:
    json.dump(selected_top_features, f, indent=2)

selector = xgb_pipeline.named_steps["feature_selection"]

with mlflow.start_run(run_name="XGBoost_Feature_Selection"):
    mlflow.log_param("feature_selection_method_1", "drop_high_missing_and_constant_features_inside_pipeline")
    mlflow.log_param("feature_selection_method_2", "xgboost_feature_importance_top_100_for_analysis")
    mlflow.log_param("max_missing_ratio", selector.max_missing_ratio)
    mlflow.log_param("min_unique_values", selector.min_unique_values)
    mlflow.log_metric("pipeline_selected_feature_count", len(selector.selected_features_))
    mlflow.log_metric("pipeline_dropped_feature_count", len(selector.dropped_features_))
    mlflow.log_metric("importance_selected_top_n", len(selected_top_features))
    mlflow.log_artifact("xgboost_feature_importance.csv")
    mlflow.log_artifact("xgboost_top_100_features.json")

print("Feature selection artifacts logged to MLflow.")
print("Selected inside pipeline:", len(selector.selected_features_))
print("Dropped inside pipeline:", len(selector.dropped_features_))


# 6. Final Pipeline and Model Registry


In [ ]:
final_xgb_params = best_params_for_final.copy()

full_negative_count = (y == 0).sum()
full_positive_count = (y == 1).sum()
full_scale_pos_weight = full_negative_count / full_positive_count

final_xgb_params["scale_pos_weight"] = full_scale_pos_weight

pipeline = Pipeline([
    ("drop_columns", DropColumns(cols=["TransactionID"])),
    ("feature_engineering", FeatureEngineering()),
    ("frequency_encoding", FrequencyEncoder(cols=freq_encode_cols)),
    ("categorical_encoding", CategoricalEncoder()),
    ("replace_inf", ReplaceInfValues()),
    ("feature_selection", SimpleFeatureSelector(max_missing_ratio=0.95, min_unique_values=2)),
    ("model", XGBClassifier(**final_xgb_params))
])

with mlflow.start_run(run_name="XGBoost_Final_Pipeline"):
    pipeline.fit(X, y)

    full_train_pred = pipeline.predict_proba(X)[:, 1]
    full_train_roc_auc = roc_auc_score(y, full_train_pred)
    full_train_pr_auc = average_precision_score(y, full_train_pred)

    mlflow.log_params(final_xgb_params)
    mlflow.log_param("model_architecture", "XGBoost")
    mlflow.log_param("model_saved_as", "sklearn_pipeline")
    mlflow.log_param("registered_model_name", REGISTERED_MODEL_NAME)
    mlflow.log_param("best_params_source", best_params_source)
    mlflow.log_param("selected_validation_experiment", best_experiment_name)
    mlflow.log_metric("selected_validation_roc_auc", val_roc_auc)
    mlflow.log_metric("selected_validation_pr_auc", val_pr_auc)
    mlflow.log_metric("full_train_roc_auc", full_train_roc_auc)
    mlflow.log_metric("full_train_pr_auc", full_train_pr_auc)
    mlflow.log_metric("full_train_rows", X.shape[0])
    mlflow.log_metric("full_train_fraud_rate", y.mean())

    with open("model.pkl", "wb") as f:
        pickle.dump(pipeline, f)
    mlflow.log_artifact("model.pkl")

    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        registered_model_name=REGISTERED_MODEL_NAME
    )

final_pipeline = pipeline

print("Final pipeline trained and registered.")
print("Selected validation experiment:", best_experiment_name)
print("Full-train ROC-AUC sanity check:", full_train_roc_auc)
print("Full-train PR-AUC sanity check:", full_train_pr_auc)


In [ ]:
test_pred_proba = final_pipeline.predict_proba(X_test)[:, 1]

In [ ]:
test_ids = test["TransactionID"].copy()
submission = pd.DataFrame({
    "TransactionID": test_ids,
    "isFraud": test_pred_proba
})


In [ ]:
submission.to_csv("submission.csv", index=False)

submission.head()

In [ ]:
with mlflow.start_run(run_name="XGBoost_Submission"):
    mlflow.log_metric("submission_rows", submission.shape[0])
    mlflow.log_artifact("submission.csv")

print("Submission artifact logged to MLflow.")
